In [ ]:
# Parameters — edit these before running
REPO_URL    = "https://github.com/Dev252001/ham10000-lesion-eval.git"
REPO_BRANCH = "main"  # set to another branch to test before merging
SEED        = 42
SPLIT_MODE  = "lesion"  # "image" is for the leakage comparison

In [ ]:
# Setup: mount Drive, clone / checkout / pull repo, install extras
# No GPU needed for this notebook — only reads JSON and predictions.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, subprocess, sys
from pathlib import Path

DRIVE_DIR  = Path("/content/drive/MyDrive/ham10000_project")
REPO_DIR   = DRIVE_DIR / "ham10000-lesion-eval"
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR / "src"))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pandas", "scikit-learn", "matplotlib"],
    check=True,
)

In [ ]:
# Load metrics JSONs — no model loading, no GPU
import json
from pathlib import Path

metrics_dir = REPO_DIR / "outputs" / "metrics"
preds_dir   = REPO_DIR / "outputs" / "predictions"
figures_dir = REPO_DIR / "outputs" / "figures"

json_files = sorted(metrics_dir.glob("*.json"))
if not json_files:
    print("No metrics found. Run notebooks 02 and 03 first.")

runs = {}
for jf in json_files:
    data = json.loads(jf.read_text(encoding="utf-8"))
    runs[jf.stem] = data
    print(f"{jf.stem}: macro_f1={data['metrics'].get('macro_f1', 'n/a')}")

In [ ]:
# Bootstrap CIs for macro-F1 and malignant recall
import pandas as pd
from skinlesion.evaluate import cluster_bootstrap_ci

ci_results = {}
for stem, data in runs.items():
    pred_file = preds_dir / f"{stem}.csv"
    if not pred_file.exists():
        print(f"Predictions not found for {stem}, skipping CI.")
        continue
    pred_df = pd.read_csv(pred_file)
    ci = {}
    for metric in ["macro_f1", "mel_recall", "bcc_recall", "akiec_recall"]:
        ci[metric] = cluster_bootstrap_ci(pred_df, metric, n_boot=1000, seed=0)
    ci_results[stem] = ci
    print(f"{stem}: macro_f1 CI = {ci['macro_f1']}")

In [ ]:
# Comparison table
import pandas as pd
from skinlesion.data import CLASSES

rows = []
for stem, data in runs.items():
    m = data.get("metrics", {})
    ci = ci_results.get(stem, {})
    mf1_ci = ci.get("macro_f1", {})
    mf1_ci_s = (
        f"({mf1_ci['ci_lower']:.3f}, {mf1_ci['ci_upper']:.3f})"
        if mf1_ci else "—"
    )
    tt = data.get("train_time_s")
    tt_s = f"{tt:.0f} s" if tt is not None else "—"
    rows.append({
        "Run": stem,
        "Accuracy": f"{m.get('accuracy', float('nan')):.4f}",
        "Macro-F1": f"{m.get('macro_f1', float('nan')):.4f}",
        "Macro-F1 CI": mf1_ci_s,
        "mel recall": f"{m.get('per_class', {}).get('mel', {}).get('recall', float('nan')):.4f}",
        "bcc recall": f"{m.get('per_class', {}).get('bcc', {}).get('recall', float('nan')):.4f}",
        "akiec recall": f"{m.get('per_class', {}).get('akiec', {}).get('recall', float('nan')):.4f}",
        "Parameters": data.get("n_params", "—"),
        "Train time": tt_s,
    })

table = pd.DataFrame(rows)
print(table.to_string(index=False))

In [ ]:
# Leakage comparison (image-mode runs, if present)
image_runs = {k: v for k, v in runs.items() if "_image_" in k}
lesion_runs = {k: v for k, v in runs.items() if "_lesion_" in k}

if image_runs and lesion_runs:
    print("Leakage comparison (image split vs lesion split)")
    for stem_i, data_i in image_runs.items():
        model = data_i.get("model_name", stem_i)
        seed  = data_i.get("seed", "?")
        stem_l_candidates = [k for k in lesion_runs
                              if lesion_runs[k].get("model_name") == model
                              and lesion_runs[k].get("seed") == seed]
        for stem_l in stem_l_candidates:
            mf1_i = data_i["metrics"].get("macro_f1", float("nan"))
            mf1_l = lesion_runs[stem_l]["metrics"].get("macro_f1", float("nan"))
            print(f"  {model} seed={seed}: image={mf1_i:.4f}  lesion={mf1_l:.4f}  diff={mf1_i - mf1_l:+.4f}")
else:
    print("Image-mode runs not found; run notebooks 02-03 with SPLIT_MODE='image' to compare.")

In [ ]:
# Bar chart: macro-F1 per model
import matplotlib.pyplot as plt

labels = [r["Run"] for r in rows]
mf1_vals = [float(r["Macro-F1"]) for r in rows]

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.5), 4))
ax.bar(labels, mf1_vals, color="steelblue")
ax.set_ylim(0, 1)
ax.set_ylabel("Macro-F1")
ax.set_title("Macro-F1 by run")
ax.set_xticklabels(labels, rotation=20, ha="right")
fig.tight_layout()
comp_fig_path = figures_dir / "comparison_macro_f1.png"
figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(comp_fig_path, dpi=100)
plt.close(fig)
print(f"Saved {comp_fig_path}")

In [ ]:
# Package outputs and download
from skinlesion.export import make_outputs_zip

zip_path = DRIVE_DIR / "outputs_05.zip"
make_outputs_zip(REPO_DIR, zip_path)
from google.colab import files
files.download(str(zip_path))
print("Unzip at the root of your local clone, then commit.")